In [1]:
import numpy as np
import scipy as sp

In [2]:
def mc_price(S0, K, T, r, sigma, M):
    """
    Monte Carlo pricer for European Call/Put options
    """

    rng = np.random.default_rng()

    random_array = rng.standard_normal(M)

    S_array = S0 * np.exp( ( r - sigma ** 2 / 2 ) * T + random_array * sigma * np.sqrt(T) )

    discount = np.exp( - r * T )
    
    c_array = discount * np.maximum( S_array[:, np.newaxis] - K , 0 )
    p_array = discount * np.maximum( K - S_array[:, np.newaxis] , 0 )

    c_se = np.std( c_array, axis=0, ddof=1 ) / np.sqrt( M )
    p_se = np.std( p_array, axis=0, ddof=1 ) / np.sqrt( M )

    return c_array.mean( axis=0 ), p_array.mean( axis=0 ), c_se, p_se

In [21]:
S0 = 100.0
K = np.array([90, 100, 110]) 
T = 1.0
r = 0.05
sigma = 0.20
M = 1000000

c, p, c_se, p_se = mc_price(S0, K, T, r, sigma, M)
print("Calls:", c, "SE:", c_se)
print("Puts:", p, "SE:", p_se)

Calls: [16.69632298 10.44977998  6.0378269 ] SE: [0.01737516 0.01470116 0.01161358]
Puts: [ 2.3252637   5.59101495 10.69135612] SE: [0.0053722  0.00868501 0.01201761]


In [17]:
def mc_price_antithetic(S0, K, T, r, sigma, M):
    """
    Monte Carlo pricer for European Call/Put options using antithetic variates
    """

    rng = np.random.default_rng()
    
    random_array = rng.standard_normal( int( M / 2 ))
    neg_random_array = ( - 1 ) * random_array

    S1_array = S0 * np.exp( ( r - sigma ** 2 / 2 ) * T + random_array * sigma * np.sqrt(T) )
    S2_array = S0 * np.exp( ( r - sigma ** 2 / 2 ) * T + neg_random_array * sigma * np.sqrt(T) )

    discount = np.exp( - r * T )

    c1_array = discount * np.maximum( S1_array[:, np.newaxis] - K , 0 )
    c2_array = discount * np.maximum( S2_array[:, np.newaxis] - K , 0 )
    p1_array = discount * np.maximum( K - S1_array[:, np.newaxis] , 0 )
    p2_array = discount * np.maximum( K - S2_array[:, np.newaxis] , 0 )

    c_array = ( c1_array + c2_array ) / 2
    p_array = ( p1_array + p2_array ) / 2

    c_se = np.std( c_array, axis=0, ddof=1 ) / np.sqrt( M / 2 )
    p_se = np.std( p_array, axis=0, ddof=1 ) / np.sqrt( M / 2 )

    return c_array.mean( axis=0 ), p_array.mean( axis=0 ), c_se, p_se

In [22]:
S0 = 100.0
K = np.array([90, 100, 110]) 
T = 1.0
r = 0.05
sigma = 0.20
M = 1000000

c, p, c_se, p_se = mc_price_antithetic(S0, K, T, r, sigma, M)
print("Calls:", c, "SE:", c_se)
print("Puts:", p, "SE:", p_se)

Calls: [16.71329019 10.46406828  6.05390053] SE: [0.00876567 0.01041719 0.00996298]
Puts: [ 2.31820613  5.58127847 10.68340496] SE: [0.00483011 0.00663534 0.00612058]
